# SHAP analysis of the Mannich enantioselectivity models (DFT set A + reaction conditions)

Post-hoc explanation of the models from `mannich_regression.ipynb` and `mannich_classification.ipynb`.

* Representation: DFT descriptor set A (M06-2X/def2-TZVP // r2SCAN-3c) + reaction conditions (56 features).
* Models: XGBoost (primary, all folds) and random forest (consistency check, a subset of folds).
* Splits: `random` (5-fold x 5 repeats) and `gkf10_catalyst`. `lorgo_group` is not explained: all models are at the dummy level there.
* Models are **refit** with the same estimators and preprocessing as the original notebooks (training-fold median imputation, constant columns dropped, `tree_method='hist'`, balanced class weights for the classifiers), the saved folds, tuned hyperparameters and per-fold seeds — the run cache stores predictions only. The refit is checked against `metrics_summary.csv` and the stored out-of-fold predictions.
* SHAP: TreeSHAP (`tree_path_dependent`) on the **test fold** of every split. Regression: kcal/mol. Classification: XGBoost in log-odds, RF in probability of class 1 (ee >= threshold).
* Global importance = mean |SHAP| over test reactions of one fold, then mean +- SD over folds.

Inputs are only read. All outputs go to new folders `shap_regression/` and `shap_classification/`.

In [ ]:
import os, json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import shap, xgboost, sklearn
from joblib import Parallel, delayed
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, roc_auc_score, average_precision_score, matthews_corrcoef
warnings.filterwarnings('ignore', category=UserWarning)
print('shap', shap.__version__, '| xgboost', xgboost.__version__, '| sklearn', sklearn.__version__)

In [ ]:
# ---- paths (edit if needed) ----
BASE = Path('..')                         # repository root, relative to notebooks/
DATA_PATH = BASE / 'data/processed/mannich_dataset_hbd.csv'
REPR_DIR = Path(os.environ.get('SHAP_REPR_DIR', BASE / 'data/representations'))
FOLDS_DIR = BASE / 'splits'
REG_DIR  = Path(os.environ.get('SHAP_REG_DIR',  BASE / 'results/regression'))
CLF_DIR  = Path(os.environ.get('SHAP_CLF_DIR',  BASE / 'results/classification'))
OUT_REG  = Path(os.environ.get('SHAP_OUT_REG',  BASE / 'results/shap/shap_regression'))
OUT_CLF  = Path(os.environ.get('SHAP_OUT_CLF',  BASE / 'results/shap/shap_classification'))

# ---- settings ----
SEED = 0                                  # per-fold seed = SEED + 1000*repeat + fold (as in the run manifests)
SPLITS = ['random', 'gkf10_catalyst']
THRESHOLDS = [70, 80, 90]
MODELS = ['xgb', 'rf']
RF_RANDOM_REPEATS = [0]                   # RF is the consistency check: repeat 0 of the random split + all gkf folds
N_WORKERS = int(os.environ.get('SHAP_N_WORKERS', max(1, (os.cpu_count() or 2) - 1)))
TOP_N = 20
CORR_CUT = 0.8                            # |Spearman rho| to flag correlated descriptors
for p in (OUT_REG, OUT_CLF):
    (p / 'figures').mkdir(parents=True, exist_ok=True); (p / 'cache').mkdir(exist_ok=True)

GROUP_ORDER = ['catalyst', 'electrophile', 'nucleophile', 'solvent', 'conditions']
GROUP_COLORS = dict(zip(GROUP_ORDER, ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']))
def group_of(c):
    return {'cat': 'catalyst', 'ele': 'electrophile', 'nuc': 'nucleophile', 'solv': 'solvent', 'cond': 'conditions'}[c.split('_')[0]]
plt.rcParams.update({'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': '#c3c2b7', 'axes.grid': False, 'savefig.dpi': 300, 'pdf.fonttype': 42})
def savefig(fig, out, name):
    for ext in ('pdf', 'png'):
        fig.savefig(out / 'figures' / f'{name}.{ext}', bbox_inches='tight')

## 1. Data, folds, hyperparameters

In [ ]:
data = pd.read_csv(DATA_PATH)
dft  = pd.read_csv(REPR_DIR / 'features_dft.csv')
cond = pd.read_csv(REPR_DIR / 'features_conditions.csv')
X = dft.merge(cond, on='reaction_id', validate='1:1').set_index('reaction_id').loc[data.reaction_id]
FEATS = list(X.columns); GROUPS = {f: group_of(f) for f in FEATS}
y_reg = data.set_index('reaction_id').loc[X.index, 'ddG_kcal_mol']
ee_abs = data.set_index('reaction_id').loc[X.index, 'ee'].abs()
folds = {s: pd.read_csv(FOLDS_DIR / f'folds_{s}.csv') for s in SPLITS}   # shared by regression and classification
hp_reg = pd.read_csv(REG_DIR / 'hyperparameters_tuned.csv').query("representation == 'dft'")
hp_clf = pd.read_csv(CLF_DIR / 'hyperparameters_tuned.csv').query("representation == 'dft'")
def get_hp(task, split, model, thr=None):
    t = hp_reg if task == 'reg' else hp_clf.query('threshold == @thr')
    return json.loads(t.query('split == @split and model == @model').params.iloc[0])
print(X.shape, '| NaN per column:', X.isna().sum()[X.isna().sum() > 0].to_dict())
print(pd.Series(GROUPS).value_counts().to_dict())

## 2. Refit + TreeSHAP per fold

In [ ]:
def fold_list(split, model):
    fd = folds[split]
    for (r, k), g in fd.groupby(['repeat', 'fold']):
        if split == 'random' and model == 'rf' and r not in RF_RANDOM_REPEATS:
            continue
        te = g.reaction_id.values
        tr = fd[(fd.repeat == r) & (fd.fold != k)].reaction_id.values
        yield int(r), int(k), tr, te

def make_model(task, model, params, seed, y_train=None):
    """Same estimators as in mannich_regression / mannich_classification."""
    if task == 'reg':
        if model == 'xgb':
            return XGBRegressor(**params, tree_method='hist', random_state=seed, n_jobs=1, verbosity=0)
        return RandomForestRegressor(**params, random_state=seed, n_jobs=1)
    if model == 'xgb':   # BalancedXGBClassifier: scale_pos_weight = n0 / n1 on the training rows
        n1 = int(y_train.sum()); spw = (len(y_train) - n1) / max(n1, 1)
        return XGBClassifier(**params, scale_pos_weight=spw, objective='binary:logistic', eval_metric='logloss',
                             tree_method='hist', random_state=seed, n_jobs=1, verbosity=0)
    return RandomForestClassifier(**params, class_weight='balanced', random_state=seed, n_jobs=1)

def block_prep(Xtr, Xte):
    """BlockPreprocessor(n_fp=0, scale_continuous=False): training-fold medians, drop columns constant in training."""
    A = Xtr.to_numpy(float); B = Xte.to_numpy(float)
    med = np.nanmedian(A, axis=0); med = np.where(np.isnan(med), 0.0, med)
    A = np.where(np.isnan(A), med, A); B = np.where(np.isnan(B), med, B)
    keep = A.max(axis=0) > A.min(axis=0)
    return A[:, keep], B[:, keep], keep

def run_fold(task, split, model, thr, r, k, tr, te):
    y = y_reg if task == 'reg' else (ee_abs >= thr).astype(int)
    A, B, keep = block_prep(X.loc[tr], X.loc[te])
    m = make_model(task, model, get_hp(task, split, model, thr), SEED + 1000 * r + k, y.loc[tr].values).fit(A, y.loc[tr].values)
    sv = shap.TreeExplainer(m)(B)
    vals, base = sv.values, np.asarray(sv.base_values)
    if vals.ndim == 3:                     # sklearn classifier: one output per class -> class 1
        vals, base = vals[..., 1], base[..., 1]
    if task == 'reg':
        pred = m.predict(B); out = pred
    elif model == 'xgb':
        pred = m.predict_proba(B)[:, 1]; out = m.predict(B, output_margin=True)
    else:
        pred = m.predict_proba(B)[:, 1]; out = pred
    add_err = float(np.abs(vals.sum(1) + base - out).max())   # SHAP additivity check
    full = np.zeros((len(te), len(FEATS))); full[:, keep] = vals   # dropped (constant) columns contribute 0
    meta = pd.DataFrame({'reaction_id': te, 'repeat': r, 'fold': k, 'y_true': y.loc[te].values, 'y_pred': pred, 'base_value': base})
    return pd.concat([meta, pd.DataFrame(full, columns=FEATS)], axis=1), add_err

def run_config(task, split, model, thr=None):
    out = OUT_REG if task == 'reg' else OUT_CLF
    tag = f'{task}_{split}_{model}' + ('' if thr is None else f'_ee{thr}')
    f = out / 'cache' / f'shap_{tag}.csv.gz'
    if f.exists():
        return pd.read_csv(f)
    t0 = time.time()
    res = Parallel(n_jobs=N_WORKERS)(delayed(run_fold)(task, split, model, thr, *fl) for fl in fold_list(split, model))
    df = pd.concat([r[0] for r in res], ignore_index=True)
    print(f'{tag}: {len(res)} folds, {time.time()-t0:.0f} s, max additivity error {max(r[1] for r in res):.1e}')
    df.to_csv(f, index=False)
    return df

CONFIGS = [('reg', s, m, None) for s in SPLITS for m in MODELS] + \
          [('clf', s, m, t) for t in THRESHOLDS for s in SPLITS for m in MODELS]
SH = {}
for c in CONFIGS:
    SH[c] = run_config(*c)

## 3. Refit check against the original runs

In [ ]:
def pooled_metrics(task, df):
    rows = []
    for r, g in df.groupby('repeat'):
        if task == 'reg':
            rows.append({'R2': r2_score(g.y_true, g.y_pred), 'MAE': mean_absolute_error(g.y_true, g.y_pred),
                         'RMSE': mean_squared_error(g.y_true, g.y_pred) ** 0.5})
        else:
            rows.append({'ROC_AUC': roc_auc_score(g.y_true, g.y_pred), 'PR_AUC': average_precision_score(g.y_true, g.y_pred),
                         'MCC': matthews_corrcoef(g.y_true, (g.y_pred >= 0.5).astype(int))})
    return pd.DataFrame(rows).mean()

ms_reg = pd.read_csv(REG_DIR / 'metrics_summary.csv').query("representation == 'dft'")
ms_clf = pd.read_csv(CLF_DIR / 'metrics_summary.csv').query("representation == 'dft'")
check = []
for (task, s, m, t), df in SH.items():
    mine = pooled_metrics(task, df)
    ref = (ms_reg.query('split == @s and model == @m') if task == 'reg' else ms_clf.query('split == @s and model == @m and threshold == @t')).iloc[0]
    row = {'task': task, 'split': s, 'model': m, 'threshold': t, 'n_repeats_refit': df.repeat.nunique()}
    for k in mine.index:
        row[f'{k}_refit'] = round(mine[k], 4); row[f'{k}_original'] = ref[k]
    check.append(row)
check = pd.DataFrame(check)

# prediction-level agreement for regression (original OOF predictions are stored)
oof = pd.read_csv(REG_DIR / 'predictions_oof.csv').query("representation == 'dft' and model in @MODELS and split in @SPLITS")
agree = []
for (task, s, m, t), df in SH.items():
    if task != 'reg': continue
    j = df.merge(oof.query('split == @s and model == @m'), on=['reaction_id', 'repeat', 'fold'], suffixes=('', '_orig'))
    agree.append({'split': s, 'model': m, 'n': len(j), 'pearson_r': np.corrcoef(j.y_pred, j.y_pred_orig)[0, 1],
                  'MAE_between_runs': mean_absolute_error(j.y_pred, j.y_pred_orig)})
clf_oof_path = CLF_DIR / 'predictions_oof.csv'
if clf_oof_path.exists():
    oofc = pd.read_csv(clf_oof_path, usecols=['reaction_id', 'threshold', 'split', 'repeat', 'fold', 'representation', 'model', 'score'])
    oofc = oofc.query("representation == 'dft' and model in @MODELS and split in @SPLITS")
    for (task, s, m, t), df in SH.items():
        if task != 'clf': continue
        j = df.merge(oofc.query('split == @s and model == @m and threshold == @t'), on=['reaction_id', 'repeat', 'fold'])
        agree.append({'split': s, 'model': m, 'threshold': t, 'n': len(j), 'pearson_r': np.corrcoef(j.y_pred, j.score)[0, 1],
                      'MAE_between_runs': mean_absolute_error(j.y_pred, j.score)})
agree = pd.DataFrame(agree)
check.to_csv(OUT_REG / 'refit_check_metrics.csv', index=False); agree.to_csv(OUT_REG / 'refit_check_predictions.csv', index=False)
display(check); display(agree)

## 4. Global importance (mean |SHAP|) and group contributions

In [ ]:
def cfg_name(c):
    task, s, m, t = c
    return f"{'regression' if task == 'reg' else f'clf ee>={t}'} | {m.upper()} | {s}"

def importance(df):
    per_fold = df.groupby(['repeat', 'fold'])[FEATS].apply(lambda g: g.abs().mean())
    out = pd.DataFrame({'mean_abs_shap': per_fold.mean(), 'sd_over_folds': per_fold.std(ddof=1), 'n_folds': len(per_fold)})
    out['share'] = out.mean_abs_shap / out.mean_abs_shap.sum()
    out['rank'] = out.mean_abs_shap.rank(ascending=False, method='min').astype(int)
    out['group'] = [GROUPS[f] for f in out.index]
    # fold-to-fold stability of the ranking: mean pairwise Spearman between fold vectors
    rho = spearmanr(per_fold.T.values).statistic
    stab = np.nanmean(rho[np.triu_indices_from(rho, 1)]) if np.ndim(rho) else rho
    return out.sort_values('rank'), per_fold, stab

def group_importance(df):
    g = pd.DataFrame({grp: df[[f for f in FEATS if GROUPS[f] == grp]].sum(1) for grp in GROUP_ORDER})
    g[['repeat', 'fold']] = df[['repeat', 'fold']]
    per_fold = g.groupby(['repeat', 'fold'])[GROUP_ORDER].apply(lambda x: x.abs().mean())
    return per_fold.mean(), per_fold.std(ddof=1)

# correlated descriptor pairs (whole data set)
rho_X = X.corr(method='spearman')
pairs = [(a, b, rho_X.loc[a, b]) for i, a in enumerate(FEATS) for b in FEATS[i+1:] if abs(rho_X.loc[a, b]) >= CORR_CUT]
corr_pairs = pd.DataFrame(pairs, columns=['feature_a', 'feature_b', 'spearman_rho']).sort_values('spearman_rho', key=abs, ascending=False)
partners = {f: ', '.join(sorted(set(corr_pairs.query('feature_a == @f').feature_b) | set(corr_pairs.query('feature_b == @f').feature_a))) for f in FEATS}

IMP, PERFOLD, STAB, GRP = {}, {}, {}, {}
for c, df in SH.items():
    IMP[c], PERFOLD[c], STAB[c] = importance(df)
    IMP[c]['correlated_with'] = [partners[f] for f in IMP[c].index]
    GRP[c] = group_importance(df)

def dump(task, out):
    long = pd.concat([IMP[c].assign(config=cfg_name(c), threshold=c[3], split=c[1], model=c[2]) for c in SH if c[0] == task])
    long.rename_axis('feature').reset_index().to_csv(out / f'importance_{task}.csv', index=False)
    grp = pd.concat([pd.DataFrame({'mean_abs_shap': GRP[c][0], 'sd_over_folds': GRP[c][1]}).assign(config=cfg_name(c)) for c in SH if c[0] == task])
    grp['share_of_sum'] = grp.mean_abs_shap / grp.groupby('config').mean_abs_shap.transform('sum')
    grp.rename_axis('group').reset_index().to_csv(out / f'group_importance_{task}.csv', index=False)
dump('reg', OUT_REG); dump('clf', OUT_CLF)
corr_pairs.to_csv(OUT_REG / 'correlated_descriptor_pairs.csv', index=False)
print(f'{len(corr_pairs)} descriptor pairs with |rho| >= {CORR_CUT}')
print('fold-to-fold ranking stability (mean pairwise Spearman):')
print(pd.Series({cfg_name(c): round(v, 3) for c, v in STAB.items()}).to_string())
display(IMP[('reg', 'random', 'xgb', None)].head(15).round(4))

## 5. Figures — regression

In [ ]:
def bar_panel(ax, imp, title, n=TOP_N, xlabel='mean |SHAP|'):
    d = imp.head(n).iloc[::-1]
    ax.barh(range(len(d)), d.mean_abs_shap, xerr=d.sd_over_folds, height=0.7,
            color=[GROUP_COLORS[g] for g in d.group], edgecolor='white', linewidth=1,
            error_kw=dict(ecolor='#5f5e5a', elinewidth=0.8, capsize=1.5))
    ax.set_yticks(range(len(d))); ax.set_yticklabels(d.index, fontsize=7.5)
    ax.set_title(title, fontsize=9, loc='left'); ax.set_xlabel(xlabel)
    ax.xaxis.grid(True, color='#e1e0d9', linewidth=0.6); ax.set_axisbelow(True)

def group_legend(fig):
    h = [plt.Rectangle((0, 0), 1, 1, color=GROUP_COLORS[g]) for g in GROUP_ORDER]
    fig.legend(h, GROUP_ORDER, loc='lower center', ncol=5, frameon=False, bbox_to_anchor=(0.5, -0.08))

for m in MODELS:
    fig, axes = plt.subplots(1, 2, figsize=(9, 6), constrained_layout=True)
    for ax, s in zip(axes, SPLITS):
        c = ('reg', s, m, None)
        bar_panel(ax, IMP[c], f'{m.upper()}, {s} (n folds = {IMP[c].n_folds.iloc[0]})', xlabel='mean |SHAP|, kcal/mol')
    group_legend(fig); savefig(fig, OUT_REG, f'reg_importance_top{TOP_N}_{m}'); plt.show()

In [ ]:
# beeswarm: each reaction explained once (random: repeat 0; gkf: its only repeat)
for s in SPLITS:
    df = SH[('reg', s, 'xgb', None)].query('repeat == 0').sort_values('reaction_id')
    ex = shap.Explanation(values=df[FEATS].values, base_values=df.base_value.values,
                          data=X.loc[df.reaction_id].values, feature_names=FEATS)
    plt.figure()
    shap.plots.beeswarm(ex, max_display=TOP_N, show=False, plot_size=(7, 7))
    fig = plt.gcf(); fig.axes[0].set_xlabel('SHAP value, kcal/mol (impact on |ΔΔG‡|)')
    fig.axes[0].set_title(f'XGBoost, {s}: test-fold SHAP values', fontsize=9, loc='left')
    savefig(fig, OUT_REG, f'reg_beeswarm_xgb_{s}'); plt.show()

In [ ]:
# dependence plots for the top descriptors (random split, XGB, all repeats pooled)
df = SH[('reg', 'random', 'xgb', None)]
top = IMP[('reg', 'random', 'xgb', None)].index[:9]
fig, axes = plt.subplots(3, 3, figsize=(9, 8), constrained_layout=True)
for ax, f in zip(axes.flat, top):
    xv = X.loc[df.reaction_id, f].values
    ax.scatter(xv, df[f], s=8, alpha=0.35, color=GROUP_COLORS[GROUPS[f]], linewidths=0)
    ax.axhline(0, color='#c3c2b7', linewidth=0.8)
    ax.set_xlabel(f, fontsize=8); ax.set_ylabel('SHAP, kcal/mol', fontsize=8)
group_legend(fig); savefig(fig, OUT_REG, 'reg_dependence_top9_xgb_random'); plt.show()

## 6. Figures — classification

In [ ]:
for m in MODELS:
    for s in SPLITS:
        fig, axes = plt.subplots(1, 3, figsize=(12, 5.5), constrained_layout=True)
        unit = 'mean |SHAP|, log-odds' if m == 'xgb' else 'mean |SHAP|, P(class 1)'
        for ax, t in zip(axes, THRESHOLDS):
            bar_panel(ax, IMP[('clf', s, m, t)], f'ee ≥ {t}%', n=15, xlabel=unit)
        fig.suptitle(f'{m.upper()} classifier, {s}', x=0.01, ha='left', fontsize=10)
        group_legend(fig); savefig(fig, OUT_CLF, f'clf_importance_top15_{m}_{s}'); plt.show()

In [ ]:
for t in THRESHOLDS:
    df = SH[('clf', 'random', 'xgb', t)].query('repeat == 0').sort_values('reaction_id')
    ex = shap.Explanation(values=df[FEATS].values, base_values=df.base_value.values,
                          data=X.loc[df.reaction_id].values, feature_names=FEATS)
    plt.figure(); shap.plots.beeswarm(ex, max_display=15, show=False, plot_size=(7, 6))
    fig = plt.gcf(); fig.axes[0].set_xlabel(f'SHAP value, log-odds of ee ≥ {t}%')
    fig.axes[0].set_title(f'XGBoost classifier ee ≥ {t}%, random', fontsize=9, loc='left')
    savefig(fig, OUT_CLF, f'clf_beeswarm_xgb_random_ee{t}'); plt.show()

In [ ]:
# combined beeswarm for the three ee thresholds (XGB, random split, repeat 0: each reaction explained once)
N_SHOW = 15
fig, axes = plt.subplots(1, 3, figsize=(16, 6.5))
for i, (ax, t) in enumerate(zip(axes, THRESHOLDS)):
    df = SH[('clf', 'random', 'xgb', t)].query('repeat == 0').sort_values('reaction_id')
    ex = shap.Explanation(values=df[FEATS].values, base_values=df.base_value.values,
                          data=X.loc[df.reaction_id].values, feature_names=FEATS)
    shap.plots.beeswarm(ex, max_display=N_SHOW, show=False, ax=ax, color_bar=(i == len(THRESHOLDS) - 1), plot_size=None)
    ax.set_xlabel(f'SHAP value, log-odds of ee ≥ {t}%', fontsize=9)
    ax.set_title(f'{"abc"[i]}   ee ≥ {t}%', fontsize=10, loc='left', fontweight='bold')
    ax.tick_params(axis='y', labelsize=8)
fig.tight_layout(w_pad=2)
savefig(fig, OUT_CLF, 'clf_beeswarm_xgb_random_combined')
plt.show()

## 7. Consistency across models, splits, thresholds and tasks

In [ ]:
names = [cfg_name(c) for c in SH]
M = pd.DataFrame({cfg_name(c): IMP[c].mean_abs_shap.reindex(FEATS) for c in SH})
rank_corr = M.corr(method='spearman')
ref = IMP[('reg', 'random', 'xgb', None)].index[:10]
top10 = pd.Series({cfg_name(c): len(set(IMP[c].index[:10]) & set(ref)) for c in SH}, name='top10_overlap_with_reg_xgb_random')
rank_corr.to_csv(OUT_REG / 'rank_consistency_spearman.csv'); top10.to_csv(OUT_REG / 'top10_overlap.csv')

fig, ax = plt.subplots(figsize=(9, 8), constrained_layout=True)
im = ax.imshow(rank_corr.values, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=90, fontsize=7)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=7)
for i in range(len(names)):
    for j in range(len(names)):
        v = rank_corr.values[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=5.5, color='white' if v > 0.6 else '#0b0b0b')
fig.colorbar(im, ax=ax, shrink=0.7, label='Spearman ρ of mean |SHAP| (56 features)')
savefig(fig, OUT_REG, 'rank_consistency_heatmap'); plt.show()
display(top10.to_frame())

In [ ]:
# group contributions: share of summed group |SHAP| per configuration (units differ between tasks -> shares)
G = pd.DataFrame({cfg_name(c): GRP[c][0] for c in SH}).T[GROUP_ORDER]
G = G.div(G.sum(1), axis=0)
fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
left = np.zeros(len(G))
for g in GROUP_ORDER:
    ax.barh(range(len(G)), G[g], left=left, color=GROUP_COLORS[g], edgecolor='white', linewidth=2, height=0.75, label=g)
    left += G[g].values
ax.set_yticks(range(len(G))); ax.set_yticklabels(G.index, fontsize=7.5); ax.invert_yaxis()
ax.set_xlim(0, 1); ax.set_xlabel('share of summed group |SHAP|')
ax.legend(ncol=5, frameon=False, loc='lower center', bbox_to_anchor=(0.5, -0.15))
savefig(fig, OUT_REG, 'group_shares_all_configs'); plt.show()
G.round(3).to_csv(OUT_REG / 'group_shares_all_configs.csv')
display(G.round(3))

In [ ]:
manifest = {'created_utc': pd.Timestamp.utcnow().isoformat(), 'shap': shap.__version__, 'xgboost': xgboost.__version__,
            'sklearn': sklearn.__version__, 'features': FEATS, 'splits': SPLITS, 'models': MODELS, 'thresholds': THRESHOLDS,
            'rf_random_repeats': RF_RANDOM_REPEATS, 'explainer': 'TreeExplainer, tree_path_dependent, test folds',
            'preprocessing': 'training-fold median imputation, columns constant in the training fold dropped (as BlockPreprocessor)', 'seed_rule': 'SEED + 1000*repeat + fold'}
for p in (OUT_REG, OUT_CLF):
    json.dump(manifest, open(p / 'manifest_shap.json', 'w'), indent=1, ensure_ascii=False)